# Phase 1: Legal Language Model Pre-training - Encoder-Decoder

## Task
Train a legal language model from scratch. Given a 512-word prompt from a legal document, the model writes the next 128 words. Scored with BERTScore F1 (main) and ROUGE-L F1.

Model: a Transformer encoder-decoder. The encoder reads the 512 tokens of the prompt and the decoder predicts the next tokens.

## Input
- Phase-1 legal text from the Hugging Face dataset (needs the HF_TOKEN secret).
- Official Phase-1 test prompts.

## Flow of the notebook (from input to output)
1. Write the helper file cs.py: data access, tokenizer, encoder-decoder model, training loop, generation and submission helpers.
2. Log in and choose the model size and training settings.
3. Stream the legal documents, train a BPE tokenizer and turn the text into tokens (train and validation).
4. Build the encoder-decoder and pre-train it (encoder reads 512 tokens, decoder predicts the next 256), then report validation loss and perplexity.
5. Install BERTScore.
6. Generate continuations on held-out text with several generation settings and score each with BERTScore.
7. Generate continuations for the test prompts with each setting and write one submission zip per setting.
8. Zip all output files.

## Output
- legal_encdec.pt (model) and tok_encdec.json (tokenizer).
- Validation perplexity and BERTScore for each generation setting.
- Submission zips for the test prompts.

In [10]:
%%writefile cs.py
# Shared helper code (saved as cs.py)
import os, re, json, glob, math, time, random, zipfile
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
DEV = "cuda" if torch.cuda.is_available() else "cpu"
REPO, PAD, BOS, EOS, SEP, CLS = "Exploration-Lab/CS-781-Capstone", 0, 1, 2, 3, 4
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"; os.makedirs(OUT, exist_ok=True)

# It reads the Hugging Face token from Kaggle secrets and returns the token string.
def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: return os.environ.get("HF_TOKEN", "")
# It logs in to Hugging Face and returns nothing.
def hf_login():
    t = hf_token()
    if t:
        from huggingface_hub import login; login(t, add_to_git_credential=False)
    else: print("no HF_TOKEN secret found: add it under Add-ons > Secrets (needed to download the competition data)")
# It takes rel (file path in the dataset) as input, searches for a file and returns its path.
def find(rel):
    for root in (".", os.environ.get("DATA_DIR", "."), "/kaggle/input"):
        h = glob.glob(f"{root}/**/{rel}", recursive=True)
        if h: return h[0]
# It takes rel (file path in the dataset) as input, gets a dataset file: local copy if present, otherwise download it and returns a local file path.
def getf(rel):
    from huggingface_hub import hf_hub_download
    return find(rel) or hf_hub_download(REPO, rel, repo_type="dataset", token=hf_token())
# It takes rel (file path in the dataset) as input, loads a JSON file and returns the loaded JSON data.
def jload(rel):
    d = json.load(open(getf(rel), encoding="utf-8")); return d

from tokenizers import ByteLevelBPETokenizer, Tokenizer
# It takes texts, V (vocabulary size) and path (file path) as input, trains a BPE tokenizer and saves it and returns nothing.
def train_tok(texts, V, path):
    t = ByteLevelBPETokenizer(trim_offsets=True); t.train_from_iterator(texts, vocab_size=V, min_frequency=2, special_tokens=["<pad>", "<bos>", "<eos>", "<sep>", "<cls>"]); t.save(path)
class Tok:
    # It takes path (file path) as input, loads the tokenizer and returns nothing.
    def __init__(s, path): s.t = Tokenizer.from_file(path); s.V = s.t.get_vocab_size()
    # It takes text as input, converts text into token ids and returns a list of token ids.
    def ids(s, text): return s.t.encode(text).ids
    # It takes text as input, converts text into token ids and returns (token ids, character offsets).
    def enc(s, text): e = s.t.encode(text); return e.ids, e.offsets
    # It takes texts as input, converts many texts into token ids at once and returns a list of token id lists.
    def batch(s, texts): return [e.ids for e in s.t.encode_batch(texts)]
    # It takes ids (token ids) as input, converts token ids back into text and returns text.
    def dec(s, ids): return s.t.decode([int(i) for i in ids if i > 4])

class Attn(nn.Module):
    # It takes d (model size) and h (number of heads) as input, creates the attention layer and returns nothing.
    def __init__(s, d, h): super().__init__(); s.h = h; s.q = nn.Linear(d, d); s.kv = nn.Linear(d, 2 * d); s.o = nn.Linear(d, d)
    # It takes x (input tensor), mem (encoder output), mask (attention mask) and causal (use causal mask) as input, runs self-attention or cross-attention and returns output vectors.
    def forward(s, x, mem=None, mask=None, causal=False):
        m = x if mem is None else mem; B, T, D = x.shape
        q = s.q(x).view(B, T, s.h, -1).transpose(1, 2); k, v = s.kv(m).view(B, m.size(1), 2, s.h, -1).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and mask is None)
        return s.o(y.transpose(1, 2).reshape(B, T, D))
class Block(nn.Module):
    # It takes d (model size), h (number of heads), cross (add cross-attention) and p (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(s, d, h, cross=False, p=0.1):
        super().__init__(); s.n1, s.sa, s.cross = nn.LayerNorm(d), Attn(d, h), cross
        if cross: s.nx, s.ca = nn.LayerNorm(d), Attn(d, h)
        s.n2, s.ff, s.dp = nn.LayerNorm(d), nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)), nn.Dropout(p)
    # It takes x (input tensor), mem (encoder output), mm (encoder padding mask), causal (use causal mask) and mask (attention mask) as input, runs one Transformer block and returns output vectors.
    def forward(s, x, mem=None, mm=None, causal=False, mask=None):
        x = x + s.dp(s.sa(s.n1(x), mask=mask, causal=causal))
        if s.cross and mem is not None: x = x + s.dp(s.ca(s.nx(x), mem, mm))
        return x + s.dp(s.ff(s.n2(x)))
class EncDec(nn.Module):
    # It takes V (vocabulary size), d (model size), h (number of heads), ne (encoder layers), nd (decoder layers), L (max length) and p (dropout rate) as input, builds the encoder-decoder model and returns nothing.
    def __init__(s, V=16000, d=640, h=10, ne=6, nd=8, L=1024, p=0.1):
        super().__init__(); s.emb, s.pos = nn.Embedding(V, d), nn.Embedding(L, d)
        s.enc = nn.ModuleList(Block(d, h, p=p) for _ in range(ne)); s.dec = nn.ModuleList(Block(d, h, True, p) for _ in range(nd)); s.ne_, s.nd_ = nn.LayerNorm(d), nn.LayerNorm(d)
        s.apply(lambda m: nn.init.normal_(m.weight, 0, 0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
    # It takes x (input tensor) as input, turns token ids into vectors with positions and returns embedding vectors.
    def embed(s, x): return s.emb(x) + s.pos(torch.arange(x.size(1), device=x.device))
    # It takes src (source token ids) as input, runs the encoder over the input text and returns (encoder output, padding mask).
    def encode(s, src):
        mm = (src != PAD)[:, None, None, :]; x = s.embed(src)
        for b in s.enc: x = b(x, mask=mm)
        return s.ne_(x), mm
    # It takes tgt (target token ids), mem (encoder output) and mm (encoder padding mask) as input, runs the decoder and returns next-token scores.
    def decode(s, tgt, mem=None, mm=None):
        x = s.embed(tgt)
        for b in s.dec: x = b(x, mem, mm, causal=True)
        return F.linear(s.nd_(x), s.emb.weight)
    # It takes src (source token ids) and tgt (target token ids) as input, runs encoder then decoder and returns next-token scores (logits).
    def forward(s, src, tgt): return s.decode(tgt, *s.encode(src))
# It takes m (model), cfg (settings) and path (file path) as input, saves the model weights and settings and returns nothing.
def save(m, cfg, path): torch.save(dict(cfg=cfg, sd=m.state_dict()), path)
# It takes path (file path) as input, loads saved model weights and settings and returns (model, settings).
def load(path): d = torch.load(path, map_location="cpu"); m = EncDec(**d["cfg"]); m.load_state_dict(d["sd"]); return m, d["cfg"]

class LoRMA(nn.Module):
    # It takes base (base linear layer) and r (rank) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(s, base, r):
        super().__init__(); s.base = base; s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(r)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(s, x): y = s.base(x); return y + (y @ s.A.t()) @ s.B.t()
class Adapter(nn.Module):
    # It takes ff (feed-forward layer), d (model size) and r (adapter size) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(s, ff, d, r): super().__init__(); s.ff, s.dn, s.up = ff, nn.Linear(d, r), nn.Linear(r, d); nn.init.zeros_(s.up.weight); nn.init.zeros_(s.up.bias)
    # It takes x (input tensor) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(s, x): y = s.ff(x); return y + s.up(F.gelu(s.dn(y)))
# It takes m (model), tech (fine-tuning method), r (rank) and dec (also change the decoder) as input, freezes the model and adds the chosen fine-tuning layers and returns the prepared model.
def peft(m, tech, r=16, dec=True):
    if tech == "full": return m
    for p in m.parameters(): p.requires_grad_(False)
    bl = list(m.enc) + (list(m.dec) if dec else [])
    # It takes par (parent layer) as input, replaces linear layers with LoRMA versions and returns nothing.
    def wrap(par):
        for n, c in par.named_children():
            if isinstance(c, nn.Linear): setattr(par, n, LoRMA(c, r))
            else: wrap(c)
    for b in bl:
        if tech == "lorma": wrap(b)
        if tech == "adapter": b.ff = Adapter(b.ff, b.n1.normalized_shape[0], r)
    if tech == "bitfit":
        for n, p in m.named_parameters(): p.requires_grad_(n.endswith("bias") or ".n" in n)
    return m
# It takes m (model) as input, counts trainable and total parameters and returns (trainable parameters, total parameters).
def n_train(m): return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

# It takes m (model), loss_fn (loss function), sample (batch function), steps (number of steps), lr (learning rate), ev (evaluation function), every (how often), hours (time limit in hours), acc (gradient accumulation steps), wd (weight decay), warm (warm-up steps) and log (print function) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(m, loss_fn, sample, steps, lr, ev=None, every=250, hours=5.0, acc=1, wd=0.01, warm=100, log=50):
    m.to(DEV).train(); ps = [p for p in m.parameters() if p.requires_grad]; opt = torch.optim.AdamW(ps, lr=lr, weight_decay=wd, betas=(0.9, 0.95))
    sc, t0, best, bst, run, s = torch.amp.GradScaler("cuda", enabled=DEV == "cuda"), time.time(), 1e9, None, 0.0, 0
    while s < steps:
        if s == 25: steps = min(steps, max(30, int(hours * 3600 / ((time.time() - t0) / 25))))
        f = min(1, (s + 1) / warm) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps)))
        for g in opt.param_groups: g["lr"] = lr * f
        for _ in range(acc):
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): loss = loss_fn(m, *sample()) / acc
            sc.scale(loss).backward(); run += loss.item()
        sc.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); sc.step(opt); sc.update(); opt.zero_grad(set_to_none=True); s += 1
        if s % log == 0: print(f"step {s}/{steps} loss {run/log:.4f} | {(time.time()-t0)/60:.1f} min", flush=True); run = 0.0
        if ev and (s % every == 0 or s == steps):
            v = ev(m); m.train(); print(f"   eval {v:.4f} (lower is better)", flush=True)
            if v < best: best, bst = v, {k: p.detach().cpu().clone() for k, p in m.named_parameters() if p.requires_grad}
    if bst: m.load_state_dict(bst, strict=False)
    return m.eval()

# It takes m (model), src (source token ids), pre (decoder start tokens), n (tokens to generate), temp (temperature), top_p (top-p), rep (repetition penalty), nrep (n-gram size to block) and minlen (minimum length) as input, generates text token by token and returns generated token ids (or texts).
@torch.no_grad()
def generate(m, src, pre, n=200, temp=0.0, top_p=1.0, rep=1.2, nrep=3, minlen=0):
    m.eval().to(DEV); src, out = src.to(DEV), pre.to(DEV)
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        mem, mm = m.encode(src); done = torch.zeros(len(src), dtype=torch.bool, device=DEV)
        for t in range(n):
            lg = m.decode(out, mem, mm)[:, -1].float(); lg[:, PAD] = lg[:, SEP] = lg[:, CLS] = lg[:, BOS] = -1e9
            if t < minlen: lg[:, EOS] = -1e9
            seen = torch.zeros_like(lg, dtype=torch.bool).scatter_(1, out, True); lg = torch.where(seen, torch.where(lg > 0, lg / rep, lg * rep), lg)
            if nrep:
                for r in range(len(out)):
                    w = out[r].tolist()
                    if len(w) >= nrep: lg[r, [x[-1] for x in (zip(*[w[i:] for i in range(nrep)])) if list(x[:-1]) == w[-(nrep - 1):]]] = -1e9
            if temp > 0:
                pr = F.softmax(lg / temp, -1); sp, si = pr.sort(-1, descending=True); sp[(sp.cumsum(-1) - sp) > top_p] = 0
                nx = si.gather(1, torch.multinomial(sp / sp.sum(-1, keepdim=True), 1))[:, 0]
            else: nx = lg.argmax(-1)
            nx = torch.where(done, torch.full_like(nx, PAD), nx); out = torch.cat([out, nx[:, None]], 1); done |= nx == EOS
            if done.all(): break
    return [[x for x in row[pre.size(1):] if x != PAD] for row in out.tolist()]

# It takes m (model), ids (token ids) and w (token weights) as input, averages the encoder outputs over the real tokens and returns one vector per row.
def pool(m, ids, w=None):
    h, mm = m.encode(ids); w = (mm[:, 0, 0, :] if w is None else w).to(h.dtype)[..., None]; return (h * w).sum(1) / w.sum(1).clamp(min=1)
# It takes tok (tokenizer), texts, size (chunk size), head (number of first chunks) and tail (number of last chunks) as input, cuts a long document into its first and last chunks of tokens and returns (chunks per document, real-chunk flags).
def chunk_ids(tok, texts, size=512, head=2, tail=2):
    out = []
    for ids in tok.batch(texts):
        ch = [ids[i:i + size] for i in range(0, max(len(ids), 1), size)]; ch = ch if len(ch) <= head + tail else ch[:head] + ch[-tail:]
        out.append(ch + [[EOS]] * (head + tail - len(ch)))
    return out, [[len(c) > 1 or c[0] != EOS for c in ch] for ch in out]
# It takes rows (token lists) and T (length to pad to) as input, pads token lists to the same length and returns a padded tensor of token ids.
def padto(rows, T=None):
    T = T or max(len(r) for r in rows); a = np.full((len(rows), T), PAD, np.int64)
    for i, r in enumerate(rows): a[i, :len(r)] = r[:T]
    return torch.from_numpy(a)
# It takes m (model), ch (chunk ids) and ok (real-chunk flags) as input, makes one vector per document from its chunk vectors and returns one vector per document.
def doc_vec(m, ch, ok):
    B, K, T = ch.shape; v = pool(m, ch.view(B * K, T)).view(B, K, -1); o = ok[..., None].to(v.dtype); return (v * o).sum(1) / o.sum(1).clamp(min=1)

# It takes obj (object) and tag (label) as input, writes predictions into a submission zip and returns a path of the submission zip.
def submit(obj, tag):
    p = f"{OUT}/{tag}_prediction.json"; json.dump(obj, open(p, "w"), ensure_ascii=False); z = f"{OUT}/{tag}_submission.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f: f.write(p, "prediction.json")
    assert zipfile.ZipFile(z).namelist() == ["prediction.json"]; return z
# It takes h (generated texts) and r (reference texts) as input, computes BERTScore F1 between outputs and references and returns the average F1 score.
def bertscore(h, r):
    try:
        try: import bert_score
        except ImportError: import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert_score"], check=True); import importlib; importlib.invalidate_caches()
        from bert_score import BERTScorer; global _SC
        _SC = globals().get("_SC") or BERTScorer(lang="en", batch_size=16); return float(_SC.score(h, r)[2].mean())
    except Exception as e:
        print("BERTScore unavailable (", str(e)[:60], ") - using token F1"); from collections import Counter
        f = lambda a, b: (lambda x, y: 2 * sum((x & y).values()) / max(1, sum(x.values()) + sum(y.values())))(Counter(a.split()), Counter(b.split()))
        return float(np.mean([f(a, b) for a, b in zip(h, r)]))
# It takes name as input, zips all output files into one archive and returns a path of the zip file.
def zip_all(name="ALL_outputs.zip"):
    z = f"{OUT}/{name}"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
        for p in sorted(glob.glob(f"{OUT}/*")):
            if os.path.isfile(p) and not p.endswith(name) and not p.endswith(".npy"): f.write(p, os.path.basename(p))
    print(z, f"{os.path.getsize(z)/1e6:.1f} MB"); return z

Overwriting cs.py


In [11]:
# Setup: log in and choose model size and training settings
import os, sys, json, numpy as np, torch, torch.nn.functional as F
import cs as S
S.hf_login()
SMOKE = False
CFG = dict(V=16000, d=672, h=12, ne=6, nd=8, L=1024)
ENC, PRE, CONT = 512, 32, 256
STEPS, BS, ACC, LR, HOURS = 20000, 16, 2, 6e-4, 5.5
TOK_TARGET, DOC_CHARS, SAMPLE_DOCS = 160_000_000, 20000, 1500
if SMOKE: CFG = dict(V=600, d=64, h=4, ne=2, nd=2, L=1024); STEPS, BS, ACC, TOK_TARGET, SAMPLE_DOCS = 30, 4, 1, 40_000, 20

In [12]:
# Load the legal text, train tokenizer, convert text to tokens
import subprocess, sys
try: import ijson
except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ijson"], check=True); import ijson
import requests
SRC = ["Phase_1_Pretraining/Train/" + n for n in ("train_acts.json", "train_high_courts.json", "train_supreme_court.json", "train_tribunals.json")]
# It takes rel (file path in the dataset) as input, reads a big JSON file one document at a time and returns documents one by one.
def docs(rel):
    loc = S.find(rel); f = open(loc, "rb") if loc else requests.get(f"https://huggingface.co/datasets/{S.REPO}/resolve/main/{rel}", headers={"Authorization": f"Bearer {S.hf_token()}"}, stream=True, timeout=180).raw
    try:
        for d in ijson.items(f, "item"):
            if len(d.get("text", "")) > 100: yield d["text"][:DOC_CHARS]
    except Exception as e: print("stream stopped early:", type(e).__name__)
tp = f"{S.OUT}/tok_encdec.json"
S.train_tok([t for r in SRC for t, _ in zip(docs(r), range(SAMPLE_DOCS))], CFG["V"], tp); tok = S.Tok(tp)
tr, va = [], []
# It takes buf (list of documents) as input, tokenizes a group of documents and adds them to train or validation and returns an approximate number of tokens added.
def add(buf):
    for j, ids in enumerate(tok.batch(buf)): (va if j == 0 else tr).append(np.array(ids + [S.EOS], np.uint16))
    return sum(len(x) for x in buf) // 4
for r in SRC:
    n, buf = 0, []
    for t in docs(r):
        buf.append(t)
        if len(buf) == 200: n += add(buf); buf = []
        if n >= TOK_TARGET // 4: break
    if buf and n < TOK_TARGET // 4: n += add(buf)
    print(r.split("/")[-1], f"~{n/1e6:.1f}M tokens")
tr, va = np.concatenate(tr), np.concatenate(va); print(f"train {len(tr)/1e6:.1f}M tokens | validation {len(va)/1e6:.2f}M tokens")

train_high_courts.json ~40.1M tokens
train_supreme_court.json ~40.6M tokens
train_tribunals.json ~40.0M tokens
train 143.8M tokens | validation 0.68M tokens


In [13]:
# Build the encoder-decoder model and pre-train it
m = S.EncDec(**CFG); print(f"{sum(p.numel() for p in m.parameters())/1e6:.1f}M parameters")
rng = np.random.default_rng(0)
# It takes a (token array), k (number of windows) and rg (random generator) as input, cuts random training windows from the token stream and returns a list of windows.
def windows(a, k, rg=rng):
    w = np.stack([a[i:i + ENC + CONT] for i in rg.integers(0, len(a) - ENC - CONT - 1, k)]).astype(np.int64); w = torch.from_numpy(w).to(S.DEV)
    return w[:, :ENC], w[:, ENC - PRE:-1], w[:, ENC - PRE + 1:]
# It takes m (model), src (source token ids), din (decoder input) and y (labels) as input, computes the training loss for one batch and returns a loss value.
def lossf(m, src, din, y): lg = m(src, din)[:, PRE - 1:]; return F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), y[:, PRE - 1:].reshape(-1))
vb = [windows(va, 16, np.random.default_rng(1)) for _ in range(4)]
# It takes m (model) as input, computes the validation loss and returns the average validation loss.
@torch.no_grad()
def ev(m): m.eval(); return float(np.mean([float(lossf(m, *b)) for b in vb]))
S.fit(m, lossf, lambda: windows(tr, BS), STEPS, LR, ev=ev, every=1000, hours=HOURS, acc=ACC, warm=500)
S.save(m, CFG, f"{S.OUT}/legal_encdec.pt"); print(f"validation loss {ev(m):.3f} | perplexity {np.exp(ev(m)):.1f}")

101.9M parameters
step 50/20000 loss 8.4597 | 0.6 min
step 100/20000 loss 6.9636 | 1.3 min
step 150/20000 loss 6.2197 | 1.9 min
step 200/20000 loss 5.7930 | 2.6 min
step 250/20000 loss 5.4669 | 3.2 min
step 300/20000 loss 5.2622 | 3.8 min
step 350/20000 loss 5.0985 | 4.5 min
step 400/20000 loss 4.9345 | 5.1 min
step 450/20000 loss 4.7836 | 5.8 min
step 500/20000 loss 4.6698 | 6.4 min
step 550/20000 loss 4.5430 | 7.0 min
step 600/20000 loss 4.4124 | 7.7 min
step 650/20000 loss 4.2824 | 8.3 min
step 700/20000 loss 4.2048 | 9.0 min
step 750/20000 loss 4.1073 | 9.6 min
step 800/20000 loss 4.0003 | 10.2 min
step 850/20000 loss 3.9625 | 10.9 min
step 900/20000 loss 3.8798 | 11.5 min
step 950/20000 loss 3.8542 | 12.2 min
step 1000/20000 loss 3.7624 | 12.8 min
   eval 3.5467 (lower is better)
step 1050/20000 loss 3.6802 | 13.5 min
step 1100/20000 loss 3.6602 | 14.1 min
step 1150/20000 loss 3.5756 | 14.8 min
step 1200/20000 loss 3.6071 | 15.4 min
step 1250/20000 loss 3.5628 | 16.0 min
step 1300

In [16]:
# Install BERTScore and clean old files
%pip install -q bert_score
import importlib, glob, os; importlib.invalidate_caches()
from bert_score import BERTScorer
for f in glob.glob(f"{S.OUT}/p1_dev*"): os.remove(f)

Note: you may need to restart the kernel to use updated packages.


In [17]:
# Try different text generation settings and score them
# It takes prompts and **kw (extra options) as input, generates 128-word continuations for prompts and returns a list of 128-word texts.
def cont(prompts, **kw):
    src = S.padto([p[-ENC:] for p in prompts]); pre = torch.tensor([p[-PRE:] for p in prompts])
    return [" ".join(tok.dec(o).split()[:128]) for o in S.generate(m, src, pre, n=CONT, minlen=CONT // 2, **kw)]
dw = [va[i:i + ENC + 260].astype(np.int64).tolist() for i in range(0, 40 * 2000, 2000) if i + ENC + 260 < len(va)]
ref = [" ".join(tok.dec(w[ENC:]).split()[:128]) for w in dw]
DEC = [dict(temp=0.0, rep=1.2), dict(temp=0.0, rep=1.0), dict(temp=0.7, top_p=0.9, rep=1.1), dict(temp=0.5, top_p=0.9, rep=1.2)]
score = [S.bertscore(sum([cont([w[:ENC] for w in dw[i:i + 16]], **kw) for i in range(0, len(dw), 16)], []), ref) for kw in DEC]
for kw, s in zip(DEC, score): print(f"BERTScore F1 {s:.4f} | {kw}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 0.8312 | {'temp': 0.0, 'rep': 1.2}
BERTScore F1 0.8347 | {'temp': 0.0, 'rep': 1.0}
BERTScore F1 0.8317 | {'temp': 0.7, 'top_p': 0.9, 'rep': 1.1}
BERTScore F1 0.8310 | {'temp': 0.5, 'top_p': 0.9, 'rep': 1.2}


In [19]:
# Generate test predictions and make submission zips
from huggingface_hub import list_repo_files
tf = [f for f in list_repo_files(S.REPO, repo_type="dataset", token=S.hf_token()) if f.startswith("Phase_1_Pretraining/Test/")]
print(tf)
test = S.jload(tf[0]); ids = list(test)
P = [tok.ids(test[i]["text"] if isinstance(test[i], dict) else test[i]) for i in ids]
for kw, s in zip(DEC, score):
    out = sum([cont(P[i:i + 25], **kw) for i in range(0, len(P), 25)], [])
    print(S.submit({i: {"text": t} for i, t in zip(ids, out)}, f"p1_dev{s:.4f}_t{kw['temp']}_r{kw['rep']}"))

['Phase_1_Pretraining/Test/test.json']


test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

/kaggle/working/p1_dev0.8312_t0.0_r1.2_submission.zip
/kaggle/working/p1_dev0.8347_t0.0_r1.0_submission.zip
/kaggle/working/p1_dev0.8317_t0.7_r1.1_submission.zip
/kaggle/working/p1_dev0.8310_t0.5_r1.2_submission.zip


In [20]:
# Zip all outputs
S.zip_all()

/kaggle/working/ALL_outputs.zip 380.1 MB


'/kaggle/working/ALL_outputs.zip'